# Three dimensions: oxygen in the California Current

Monday's Pier record was one point measured through time. Tuesday's MOP request was one
site. Today's dataset has three spatial dimensions *and* time, and it is the first one
where the interesting question cannot be answered with a line plot.

The question is: **how deep do you have to go before the water runs out of oxygen, and
does that depth change as you move toward the coast?**

The data are a state estimate produced at Scripps: a physical–biogeochemical ocean model
fitted to the observations available in this region between 2007 and 2010. You are
visualising your own institution's model of the coastline outside the window.

## Learning objectives

By the end, you can:

- construct a bounded request against a 1.1 GB remote granule and preserve the response;
- read a NetCDF file whose stated metadata and actual contents disagree, and trust the data;
- build an interactive 3D figure and an animation from a four-dimensional array;
- derive a two-dimensional surface from a three-dimensional field and defend the threshold
  that defines it; and
- explain why a domain average can hide the signal you are looking for.

## Before you start: `plotly`

Every figure in this notebook is interactive — you rotate it, hover it, play it — and
`matplotlib` does not do that, so these three projects use `plotly` instead.

`plotly` is deliberately **not** in `environment.yml`. Nothing else in the course needs it,
and no one should have to rebuild a working environment for one optional project. The cell
below installs it into this kernel the first time you run the notebook, and does nothing on
every run after that.

In [ ]:
# plotly is not part of the course environment, so install it into this kernel on first run.
try:
    import plotly  # noqa: F401
except ModuleNotFoundError:
    %pip install --quiet "plotly>=6.0,<7"
    import plotly  # noqa: F401

print("plotly", plotly.__version__)

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import plotly.graph_objects as go
import xarray as xr

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "README.md").exists(), "Open the course project folder first."

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from climate_course.ccs import (
    CITATION,
    DEFAULT_HYPOXIC_THRESHOLD,
    dap4_subset_url,
    earthdata_opener,
    hypoxic_boundary_depth,
    load_ccs_subset,
    month_index,
)

CCS_RAW = PROJECT_ROOT / "data" / "raw" / "cms_ccs"
CCS_RAW.mkdir(parents=True, exist_ok=True)
print("Raw folder:", CCS_RAW.relative_to(PROJECT_ROOT))

## 1. Read the source before constructing the request

Open the [dataset landing page](https://www.earthdata.nasa.gov/data/catalog/ges-disc-cms-oce-bgc-ccs-1)
and the [OPeNDAP directory](https://acdisc.gesdisc.eosdis.nasa.gov/opendap/CMS/CMS_OCE_BGC_CCS.1/contents.html).

With your partner, record:

- What does **Level 4** mean, and how is that different from the Pier observations? **TODO**
- Which variables are published, and how many separate files is that? **TODO**
- What is the size of a single granule? **TODO**
- Temporal coverage and resolution: **TODO**
- One sentence you would have to write beside any figure made from these data: **TODO**

The last one matters most. These are not measurements.

## 2. One-time setup: Earthdata Login

NASA distributes these data to registered users. If you have not already:

1. Create a free account at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov/).
2. Add a line to `~/.netrc` (create the file if needed):
   `machine urs.earthdata.nasa.gov login YOUR_USERNAME password YOUR_PASSWORD`
3. Restrict it so only you can read it: `chmod 600 ~/.netrc`

Your password lives in that file and never appears in this notebook, in the repository,
or in a printed URL. The cell below confirms the credential exists without showing it.

In [ ]:
try:
    opener = earthdata_opener()
    print("Earthdata credentials found. Ready to request data.")
except (FileNotFoundError, ValueError) as error:
    print("Not ready:", error)

## 3. The granule is 1.1 GB. Your request is not.

The oxygen granule holds `time=48, depth=72, lat=171, lon=240` — about 142 million values.
You need a fraction of that, so you ask the server to cut the array before it sends it.

The cut is written as a **constraint expression** appended to the URL. Three rules that
cost us real time to discover:

- index ranges are **inclusive at both ends**, so 45 levels is `[0:44]`;
- every coordinate array must be sliced to match the data array, or the server refuses to
  build a consistent file;
- **striding is broken on this server** — asking for every second latitude returns an
  internal error, so any thinning has to happen locally after the download.

In [ ]:
july_2010 = month_index(2010, 7)
depth_levels = 45          # depth[44] = 1012.5 m

oxygen_url = dap4_subset_url(
    "O2", first_month=july_2010, last_month=july_2010, depth_levels=depth_levels
)
oxygen_file = CCS_RAW / "ccs_2010-07_O2.nc4"

print("Time index for July 2010:", july_2010)
print("\nRequest URL:\n", oxygen_url)
print("\nLocal destination:", oxygen_file.relative_to(PROJECT_ROOT))

### Request audit

Read the printed URL and point to: the variable, the time index, the depth range, the two
horizontal ranges, and the output format.

Before you run it, predict:

- How many values will the response contain? **TODO**
- At 8 bytes each, how large is that uncompressed? **TODO**
- The response arrives compressed. Will it be closer to 5 MB or 50 MB? **TODO**

In [ ]:
if oxygen_file.exists():
    print("Raw file already exists; reusing it without overwriting.")
else:
    with opener.open(oxygen_url) as response:
        content_type = response.headers.get("Content-Type", "")
        assert "netcdf" in content_type, f"Expected NetCDF, got {content_type!r}"
        oxygen_file.write_bytes(response.read())

print(f"{oxygen_file.name}: {oxygen_file.stat().st_size:,} bytes")

If that failed, read the error before changing anything.

A `Content-Type` of `text/html` means the server returned an error page instead of data —
usually a malformed constraint expression or a login problem. A 500 on a large but valid
request is sometimes just transient; try once more before rewriting the URL. After the
instructor's troubleshooting checkpoint, run `python scripts/fetch_cms_ccs.py`, or copy
the recovery file into `data/raw/cms_ccs/` and record `acquisition_method:
instructor_recovery` in your manifest.

## 4. Inspect the response before you trust it

Open the preserved file exactly as it arrived, with no decoding, and read what it claims
about itself.

In [ ]:
raw = xr.open_dataset(oxygen_file, decode_times=False)
raw

### Three things the metadata gets wrong

This is a published NASA product from a respected group, and its metadata still contains
three statements that do not match the file. Check each one yourself rather than taking
this notebook's word for it. Read the attribute **names** as carefully as their values.

In [ ]:
o2 = raw["O2"]

# Claim 1: the variable declares a fill value. Look closely at how it is spelled.
print("variable attributes:", dict(o2.attrs))
print("cells equal to -9999:", int((o2.values == -9999).sum()))
print("cells that are NaN: ", int(np.isnan(o2.values).sum()))

# Claim 2: the global attributes state the latitude bounds.
print("\ndeclared latitude range:",
      raw.attrs["SouthernmostLatitude"], "to", raw.attrs["NorthernmostLatitude"])
print("actual latitude range:", float(raw.lat.min()), "to", float(raw.lat.max()))

# Claim 3: read the *names* of the east/west attributes, then their values.
print("\nWesternmostLatitude =", raw.attrs["WesternmostLatitude"])
print("EasternmostLatitude =", raw.attrs["EasternmostLatitude"])
print("actual lon range:", float(raw.lon.min()), "to", float(raw.lon.max()))

What you should have found:

1. The fill value is declared as **`_Fillvalue`**. The CF convention spells it
   `_FillValue`, with a capital V. xarray masks the correctly spelled attribute
   automatically and ignores this one. Here it happens not to matter — there are zero
   cells equal to `-9999` and missing data are already NaN — but if the file had used the
   sentinel, that single lowercase letter would have fed `-9999` into every average you
   computed, silently.
2. The declared latitude range is wider than the actual coordinate array.
3. `WesternmostLatitude` and `EasternmostLatitude` hold **longitudes**, on a 0–360 grid.

Discuss with your partner:

- Which of the three is most dangerous, and why is it not the most obviously wrong one? **TODO**
- What is 231°E in the convention every map in this course uses? **TODO**
- You cannot fix the provider's file. Where should the correction live instead? **TODO**

The lesson is not that this dataset is bad — it is a careful product and we are using it
precisely because it is good. The lesson is that metadata is a claim, and the array is the
evidence.

## 5. Load it properly

`load_ccs_subset` does exactly two things the raw file needs: it converts longitude to
degrees east of Greenwich, and it turns the undecodable `months since 2007-01-01` time
axis into real timestamps. Open `src/climate_course/ccs.py` and find both.

The file on disk is never modified. Preserve the response; transform in code.

In [ ]:
oxygen = load_ccs_subset(oxygen_file)["O2"].squeeze("time", drop=True)

print("dimensions:", dict(oxygen.sizes))
print("units:", oxygen.attrs.get("units"))
print("longitude:", float(oxygen.lon.min()), "to", float(oxygen.lon.max()))
print("depth levels:", oxygen.depth.values[:6], "...", oxygen.depth.values[-1], "m")
print("\nfraction of the volume that is land or below the sea floor: "
      f"{float(np.isnan(oxygen).mean()):.1%}")

Notice the depth spacing: **10 m through the top 175 m**, then coarsening with depth. The
model resolves the upper ocean finely because that is where the structure is. That choice
is what makes the next figure worth making.

## 6. A plane descending through the water column

The simplest way into a 3D array is to look at one horizontal slice at a time. Instead of
45 separate maps, put each slice at its true depth and animate downward.

`stride` thins the grid before plotting. Every value ends up in the browser, so a
full-resolution animation would be a 40 MB page that scrolls like treacle.

In [ ]:
stride = 3
plane_data = oxygen.isel(lat=slice(None, None, stride), lon=slice(None, None, stride))
field = plane_data.transpose("depth", "lat", "lon").values
depths = plane_data.depth.values
lons, lats = plane_data.lon.values, plane_data.lat.values

low, high = np.nanpercentile(field, [1, 99])

def oxygen_plane(index):
    return go.Surface(
        x=lons, y=lats,
        z=np.full((lats.size, lons.size), -depths[index]),
        surfacecolor=field[index],
        colorscale="Viridis", cmin=low, cmax=high,
        colorbar=dict(title="O₂<br>(mol m⁻³)"),
        hovertemplate="%{x:.2f}°E %{y:.2f}°N<br>O₂ %{surfacecolor:.4f} mol m⁻³<extra></extra>",
    )

frames = [go.Frame(data=[oxygen_plane(k)], name=f"{depths[k]:.0f}")
          for k in range(depths.size)]

sweep = go.Figure(data=[oxygen_plane(0)], frames=frames)
sweep.update_layout(
    title="Oxygen on a plane descending through the water column, July 2010",
    height=640, margin=dict(l=0, r=0, t=48, b=0),
    scene=dict(
        xaxis=dict(title="Longitude (°E)"),
        yaxis=dict(title="Latitude (°N)"),
        zaxis=dict(title="Depth (m)", range=[-1050, 0],
                   tickvals=[0, -200, -400, -600, -800, -1000],
                   ticktext=["0", "200", "400", "600", "800", "1000"]),
        aspectmode="manual", aspectratio=dict(x=1.5, y=1.1, z=0.7),
        camera=dict(eye=dict(x=-1.6, y=-1.5, z=0.9)),
    ),
    updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=0.05, xanchor="left",
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=180, redraw=True), mode="immediate")]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
        ])],
    sliders=[dict(active=0, x=0.14, len=0.82, y=0.05,
        currentvalue=dict(prefix="Depth: ", suffix=" m"),
        steps=[dict(label=f.name, method="animate",
                    args=[[f.name], dict(frame=dict(duration=0, redraw=True),
                                         mode="immediate")]) for f in frames])],
)
sweep

Press play, then drag to rotate. Watch the colour, not the plane.

- At roughly what depth does the map stop looking uniform? **TODO**
- Which edge of the domain loses its oxygen first as you descend? **TODO**
- The holes that open up as the plane descends are not missing data. What are they? **TODO**

## 7. From 45 slices to one surface

Flipping through slices is how you explore. It is not how you answer the question, because
the answer — *the depth at which the water turns hypoxic* — is a single number at each
horizontal position. That is a 2D surface hiding inside the 3D volume.

Coastal ecologists call water hypoxic below about **60 mmol m⁻³**, which is `0.06` in this
file's units. That number is a convention chosen because many fish and invertebrates start
to suffer near it. It is not a property of the ocean, and a different threshold gives a
different surface. You will test that in a moment.

Open `src/climate_course/ccs.py` and read `hypoxic_boundary_depth`. Find the line that
returns NaN where a profile never crosses the threshold, and be ready to explain why
returning the deepest level instead would be a lie.

In [ ]:
boundary = hypoxic_boundary_depth(oxygen, threshold=DEFAULT_HYPOXIC_THRESHOLD)

ocean_cells = int(np.isfinite(oxygen.isel(depth=0)).sum())
resolved = int(np.isfinite(boundary).sum())

print(f"threshold: {DEFAULT_HYPOXIC_THRESHOLD} mol m⁻³")
print(f"shallowest boundary: {float(boundary.min()):.0f} m")
print(f"deepest boundary:    {float(boundary.max()):.0f} m")
print(f"resolved in {resolved / ocean_cells:.1%} of ocean columns")

# The subset only reaches 1012 m, so a boundary at the bottom level would mean the
# request was too shallow to contain the answer.
assert float(boundary.max()) < float(oxygen.depth.max()), (
    "The deepest boundary sits at the bottom of the request. "
    "Ask for more depth levels before trusting this surface."
)
print("\nCheck passed: the boundary is contained inside the requested depth range.")

## 8. The hypoxic boundary in three dimensions

In [ ]:
surface_stride = 2
shown = boundary.isel(lat=slice(None, None, surface_stride),
                      lon=slice(None, None, surface_stride))

hero = go.Figure(go.Surface(
    x=shown.lon.values, y=shown.lat.values,
    z=-shown.values, surfacecolor=shown.values,
    colorscale="Viridis", reversescale=True,
    colorbar=dict(title="Depth of<br>O₂ = 0.06<br>mol m⁻³ (m)"),
    hovertemplate="%{x:.2f}°E %{y:.2f}°N<br>hypoxic at %{surfacecolor:.0f} m<extra></extra>",
))
hero.update_layout(
    title="Depth of the hypoxic boundary, California Current System, July 2010",
    height=680, margin=dict(l=0, r=0, t=48, b=0),
    scene=dict(
        xaxis=dict(title="Longitude (°E)"),
        yaxis=dict(title="Latitude (°N)"),
        zaxis=dict(title="Depth (m)", range=[-1050, 0],
                   tickvals=[0, -200, -400, -600, -800, -1000],
                   ticktext=["0", "200", "400", "600", "800", "1000"]),
        aspectmode="manual", aspectratio=dict(x=1.5, y=1.1, z=0.7),
        camera=dict(eye=dict(x=-1.6, y=-1.5, z=0.9)),
    ),
)
hero

Rotate until you are looking along the coast from the south.

The surface lies deep offshore and climbs steeply toward the shore. Off northern
California it reaches within a few tens of metres of the sea surface, while at the same
latitude a few hundred kilometres offshore it sits near 400 m. Upwelling brings old,
oxygen-poor water up onto the shelf; that shallow shoulder is habitat compression, and it
is why hypoxia is a coastal management problem rather than an open-ocean curiosity.

- Read off the boundary depth at the latitude of Scripps Pier (32.87°N), nearshore and at
  129°W. **TODO**
- Change `threshold` in the previous cell to `0.04` and rerun both cells. Does the shape
  change, or only the depth? **TODO**
- The surface has holes. Before you look at the code again, say what a hole means. **TODO**

## 9. Extension lane

Finish the questions above before starting these. Each is independent.

### 9a. Nitrate curtains

The same upwelling that carries low-oxygen water shoreward carries nitrate with it. Hang
vertical sections in 3D and the nutrient supply becomes visible as a set of curtains.

This needs the nitrate subset, which is a second request.

In [ ]:
nitrate_file = CCS_RAW / "ccs_2010-07_NO3.nc4"
if not nitrate_file.exists():
    url = dap4_subset_url("NO3", first_month=july_2010, last_month=july_2010,
                          depth_levels=depth_levels)
    with opener.open(url) as response:
        nitrate_file.write_bytes(response.read())

nitrate = load_ccs_subset(nitrate_file)["NO3"].squeeze("time", drop=True)
nitrate = nitrate.transpose("depth", "lat", "lon")

mesh_lon, mesh_depth = np.meshgrid(nitrate.lon.values, -nitrate.depth.values)
lo, hi = np.nanpercentile(nitrate.values, [1, 99])

curtains = go.Figure()
for position, target in enumerate([30.0, 32.87, 34.5, 36.5, 38.0]):
    section = nitrate.sel(lat=target, method="nearest")
    actual = float(section.lat)
    curtains.add_trace(go.Surface(
        x=mesh_lon, y=np.full_like(mesh_lon, actual), z=mesh_depth,
        surfacecolor=section.values, colorscale="Viridis", cmin=lo, cmax=hi,
        showscale=position == 0, colorbar=dict(title="Nitrate<br>(mol m⁻³)"),
        name=f"{actual:.1f}°N",
        hovertemplate=f"{actual:.2f}°N %{{x:.2f}}°E<br>NO₃ %{{surfacecolor:.4f}} mol m⁻³<extra></extra>",
    ))

curtains.update_layout(
    title="Nitrate sections across the California Current System, July 2010",
    height=680, margin=dict(l=0, r=0, t=48, b=0),
    scene=dict(
        xaxis=dict(title="Longitude (°E)"),
        yaxis=dict(title="Latitude (°N)"),
        zaxis=dict(title="Depth (m)", range=[-1050, 0],
                   tickvals=[0, -200, -400, -600, -800, -1000],
                   ticktext=["0", "200", "400", "600", "800", "1000"]),
        aspectmode="manual", aspectratio=dict(x=1.5, y=1.1, z=0.7),
        camera=dict(eye=dict(x=-1.6, y=-1.5, z=0.9)),
    ),
)
curtains

### 9b. The seasonal cycle, and why the average hides it

Upwelling is seasonal, so the hypoxic boundary should breathe. The next request is the
whole of 2010, which is about 58 MB and takes longer than the others.

Then answer this before plotting: **the domain-mean boundary depth over 2010 moves by
about 10 m, almost monotonically.** Does that mean there is no seasonal cycle here?

In [ ]:
seasonal_file = CCS_RAW / "ccs_2010_O2_monthly.nc4"
if not seasonal_file.exists():
    url = dap4_subset_url("O2", first_month=month_index(2010, 1),
                          last_month=month_index(2010, 12), depth_levels=37)
    print("Requesting 12 months; this one is ~58 MB.")
    with opener.open(url) as response:
        seasonal_file.write_bytes(response.read())

seasonal = load_ccs_subset(seasonal_file)["O2"]
monthly = np.array([hypoxic_boundary_depth(seasonal.isel(time=k)).values
                    for k in range(seasonal.sizes["time"])])

coast = np.zeros(monthly.shape[1:], dtype=bool)
top = seasonal.isel(time=0, depth=0).values
for row in range(coast.shape[0]):
    wet = np.where(np.isfinite(top[row]))[0]
    if wet.size:
        coast[row, max(wet.max() - 30, 0):wet.max() + 1] = True   # ~200 km band

print("month  whole domain  nearshore band")
for k in range(12):
    print(f"  {k + 1:2d}   {np.nanmean(monthly[k]):9.0f} m {np.nanmean(np.where(coast, monthly[k], np.nan)):11.0f} m")

The domain mean is nearly flat because the open ocean, which barely changes, covers most
of the area. The nearshore band shoals through spring to a minimum in early summer and
relaxes afterwards — the upwelling season, exactly where you would expect it.

This is the same lesson as Thursday's anomalies, arriving from a different direction: an
average over the wrong region answers a question you did not ask.

The animation below therefore keeps depth as the *shape* of the surface and uses
**colour for each cell's departure from its own annual mean**. Red is shallower than usual.

In [ ]:
# Columns that are land in every month are all-NaN, and nanmean warns about them.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    annual_mean = np.nanmean(monthly, axis=0)
anomaly = monthly - annual_mean
limit = float(np.nanpercentile(np.abs(anomaly), 98))

months = [str(v)[:7] for v in seasonal.time.values]
sub = slice(None, None, 2)
lon_s = seasonal.lon.values[sub]
lat_s = seasonal.lat.values[sub]

def month_layer(k):
    return go.Surface(
        x=lon_s, y=lat_s,
        z=-monthly[k][sub, sub], surfacecolor=anomaly[k][sub, sub],
        colorscale="RdBu", cmin=-limit, cmax=limit,
        colorbar=dict(title="Anomaly (m)"),
        hovertemplate="%{x:.2f}°E %{y:.2f}°N<br>%{surfacecolor:+.0f} m<extra></extra>",
    )

season_frames = [go.Frame(data=[month_layer(k)], name=months[k]) for k in range(12)]
season = go.Figure(data=[month_layer(0)], frames=season_frames)
season.update_layout(
    title="Hypoxic boundary through 2010 (red = shallower than that cell's annual mean)",
    height=680, margin=dict(l=0, r=0, t=48, b=0),
    scene=dict(
        xaxis=dict(title="Longitude (°E)"),
        yaxis=dict(title="Latitude (°N)"),
        zaxis=dict(title="Depth (m)", range=[-700, 0],
                   tickvals=[0, -200, -400, -600], ticktext=["0", "200", "400", "600"]),
        aspectmode="manual", aspectratio=dict(x=1.5, y=1.1, z=0.7),
        camera=dict(eye=dict(x=-1.6, y=-1.5, z=0.9)),
    ),
    updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=0.05, xanchor="left",
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=520, redraw=True), mode="immediate")]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
        ])],
    sliders=[dict(active=0, x=0.14, len=0.82, y=0.05, currentvalue=dict(prefix="Month: "),
        steps=[dict(label=f.name, method="animate",
                    args=[[f.name], dict(frame=dict(duration=0, redraw=True),
                                         mode="immediate")]) for f in season_frames])],
)
season

### 9c. Your own question

`pH` is published in the same form and ships with the same grid. Corrosive, low-pH water
upwells onto the shelf alongside the low-oxygen water, which is the ocean acidification
story for this coastline.

Request it, pick a threshold you can defend from the literature, and build the surface.
The code you need is the code above with one variable name changed — which is the point.

## 10. What these figures do not show

Write two sentences on each, in your own words:

1. **These are not observations.** A state estimate is a model fitted to data. The figure
   is as smooth as the model, not as smooth as the ocean. **TODO**
2. **The threshold is a choice.** `0.06` came from ecology, not from this dataset. **TODO**
3. **Four years is not a trend.** 2007–2010 cannot separate a seasonal cycle from a decadal
   one, let alone a forced change. **TODO**
4. **The vertical scale is exaggerated.** The domain is roughly 1500 km wide and 1 km deep;
   the aspect ratio makes the structure legible and the slopes far steeper than reality. **TODO**

These figures also break the course's own `conventions/figure-conventions.md`, which
forbids in-figure titles because a publication figure always travels with a caption. An
interactive page has no caption, so the title has to carry it. Know which mode you are in:
if any of this ends up in a written report, the title comes off and becomes a caption.

## Manifest and exit ticket

Add one entry to `data/manifest.yml` for every file you downloaded. The `request_url`
field is the whole point of today: paste the exact constraint expression. Each response
also carries its own `history` attribute recording what the server was asked — compare
the two.

Citation for anything you make from these data:

> Verdy, A. and M. Mazloff (2017), Ocean Biogeochemistry in the California Current System
> 2007-2010 L4 Monthly, Greenbelt, MD, USA, Goddard Earth Sciences Data and Information
> Services Center (GES DISC), doi:10.5067/G854SWM56S7H

**Exit ticket.** In two sentences: you have a 3D field and a question. How do you decide
whether the answer is a slice, a derived surface, or an average — and what did the domain
mean cost you today?